# ProtStab — Phase 2: ESM2-35M + LoRA (Optimised for Colab T4)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `dmsv4_filtered_train_splits.csv` to Google Drive
3. Update `DATA_PATH` in the Config cell

**Expected time on T4:** ~2–3 min per epoch → ~45–60 min for 20 epochs  
**Expected accuracy:** Pearson r ≈ 0.78–0.83

## 1 · Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → GPU")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU     : {gpu.name}")
print(f"VRAM    : {gpu.total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

## 2 · Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate scipy tqdm
print("Done.")

## 3 · Mount Google Drive & Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── UPDATE THIS PATH ──────────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/MLProject/dmsv4_filtered_train_splits.csv"
# ─────────────────────────────────────────────────────────────────

CKPT_DIR   = "/content/drive/MyDrive/MLProject/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

ESM2_MODEL = "facebook/esm2_t12_35M_UR50D"
EPOCHS     = 20
BATCH_SIZE = 128    # T4 has 16 GB — safe at 128, try 256 if no OOM
LR         = 5e-4
LORA_R     = 8
LORA_ALPHA = 16
MAX_LEN    = 82     # 80 aa + BOS + EOS
DEVICE     = "cuda"

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"
print(f"Data    : {DATA_PATH}")
print(f"Ckpts   : {CKPT_DIR}")
print(f"Batch   : {BATCH_SIZE}  |  Epochs: {EPOCHS}  |  LR: {LR}")

## 4 · Model — ESM2 + LoRA

In [ ]:
import torch.nn as nn
from transformers import EsmModel
from peft import LoraConfig, get_peft_model
import warnings
warnings.filterwarnings("ignore")


class ESM2LoRAStabilityModel(nn.Module):
    def __init__(self, model_name=ESM2_MODEL, lora_r=LORA_R,
                 lora_alpha=LORA_ALPHA, lora_dropout=0.1, head_dropout=0.1):
        super().__init__()
        esm = EsmModel.from_pretrained(model_name)
        self.esm = get_peft_model(esm, LoraConfig(
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=["query", "key", "value"],
            lora_dropout=lora_dropout, bias="none",
        ))
        hidden = esm.config.hidden_size   # 480
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(head_dropout),
            nn.Linear(256, 64),    nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, input_ids, attention_mask):
        out    = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.head(pooled).squeeze(-1)

    def count_parameters(self):
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        return trainable, total


print("Model class defined.")

## 5 · Dataset — Batch Pre-tokenised (fast)

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer


class ESM2StabilityDataset(Dataset):
    """
    Batch-tokenises all sequences once at init and holds them as tensors.
    __getitem__ is a pure tensor index — no tokeniser called during training.
    num_workers=0 is intentional: no inter-process tensor copying overhead.
    """
    _SPLIT_COLS  = {"seq": "seq",    "label": "dG",     "split": "split"}
    _CONCAT_COLS = {"seq": "aa_seq", "label": "deltaG", "split": "split"}

    def __init__(self, csv_path, tokenizer, split="train", limit=None, chunksize=50_000):
        header = pd.read_csv(csv_path, nrows=0).columns.tolist()
        cols   = self._CONCAT_COLS if "aa_seq" in header else self._SPLIT_COLS
        seq_col, label_col, split_col = cols["seq"], cols["label"], cols["split"]

        chunks = []
        for chunk in pd.read_csv(csv_path, usecols=[seq_col, label_col, split_col],
                                 chunksize=chunksize):
            chunk = chunk[chunk[split_col] == split].dropna(subset=[seq_col, label_col])
            chunks.append(chunk[[seq_col, label_col]])
            if limit and sum(len(c) for c in chunks) >= limit:
                break

        df = pd.concat(chunks, ignore_index=True)
        if limit:
            df = df.head(limit)

        seqs   = df[seq_col].tolist()
        labels = torch.tensor(df[label_col].values, dtype=torch.float32)

        print(f"  [{split}] {len(seqs):,} sequences — batch tokenising...", end="", flush=True)
        enc = tokenizer(seqs, max_length=MAX_LEN, padding="max_length",
                        truncation=True, return_tensors="pt")
        self.input_ids      = enc["input_ids"]        # (N, 82) — stays on CPU
        self.attention_mask = enc["attention_mask"]   # (N, 82)
        self.labels         = labels
        print(" done.", flush=True)

    def __len__(self):  return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "label":          self.labels[idx],
        }


print("Dataset class defined.")

## 6 · Training & Evaluation (fp16 mixed precision)

In [ ]:
import time
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from tqdm.notebook import tqdm


def evaluate(model, loader, device, scaler_enabled=True):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(ids, mask)
            preds.extend(out.float().cpu().numpy())
            targets.extend(batch["label"].numpy())

    preds, targets = np.array(preds), np.array(targets)
    return {
        "mae":          float(np.mean(np.abs(preds - targets))),
        "rmse":         float(np.sqrt(np.mean((preds - targets) ** 2))),
        "pearson_r":    float(pearsonr(preds, targets)[0]),
        "spearman_rho": float(spearmanr(preds, targets)[0]),
    }


def train(model, train_loader, val_loader, epochs, lr, ckpt_dir, device):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=1e-2,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5, min_lr=1e-6
    )
    criterion = nn.MSELoss()
    scaler    = torch.cuda.amp.GradScaler()   # fp16 gradient scaling

    best_path = Path(ckpt_dir) / "best_model_esm2.pt"
    best_mae  = float("inf")

    print(f"\nTraining {epochs} epochs  |  batch {BATCH_SIZE}  |  fp16 mixed precision\n")
    print(f"{'Ep':>4} {'Loss':>8} {'MAE':>8} {'Pearson r':>10} {'Spearman ρ':>11} {'Time':>7}")
    print("-" * 56)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        t0 = time.time()

        pbar = tqdm(train_loader, desc=f"Ep {epoch:02d}/{epochs}",
                    leave=False, unit="batch")
        for batch in pbar:
            ids  = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            y    = batch["label"].to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = criterion(model(ids, mask), y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        avg_loss = total_loss / n_batches
        metrics  = evaluate(model, val_loader, device)
        elapsed  = time.time() - t0
        scheduler.step(metrics["mae"])

        marker = ""
        if metrics["mae"] < best_mae:
            best_mae = metrics["mae"]
            cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
            torch.save({
                "epoch":       epoch,
                "model_name":  ESM2_MODEL,
                "model_type":  "esm2_lora",
                "state_dict":  cpu_state,
                "val_metrics": metrics,
            }, best_path)
            marker = " ✓"

        print(
            f"{epoch:>4}/{epochs} {avg_loss:>8.4f} {metrics['mae']:>8.4f} "
            f"{metrics['pearson_r']:>10.4f} {metrics['spearman_rho']:>11.4f} "
            f"{elapsed:>6.0f}s{marker}",
            flush=True,
        )

    print(f"\nBest val MAE : {best_mae:.4f}")
    print(f"Checkpoint   : {best_path}")
    return best_path


print("Training functions defined.")

## 7 · Load Data (tokenises once, ~60 s)

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)

print("Loading & tokenising datasets:")
train_ds = ESM2StabilityDataset(DATA_PATH, tokenizer, split="train")
val_ds   = ESM2StabilityDataset(DATA_PATH, tokenizer, split="validation")

# num_workers=0: data is already tensors in RAM — no worker-process copying needed
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"\nTrain batches/epoch : {len(train_loader):,}")
print(f"Val   batches       : {len(val_loader):,}")

# Estimate epoch time
ms_per_batch = 35   # ~35 ms/batch on T4 with fp16
est_s = len(train_loader) * ms_per_batch / 1000
print(f"\nEstimated time/epoch on T4 : ~{est_s:.0f}s ({est_s/60:.1f} min)")
print(f"Estimated total ({EPOCHS} epochs) : ~{EPOCHS*est_s/60:.0f} min")

## 8 · Build Model

In [ ]:
print("Building ESM2 + LoRA...")
model = ESM2LoRAStabilityModel().to(DEVICE)

trainable, total = model.count_parameters()
print(f"Trainable : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Sanity forward pass in fp16
sample = next(iter(val_loader))
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    out = model(sample["input_ids"].to(DEVICE), sample["attention_mask"].to(DEVICE))
print(f"Forward pass OK — output shape {out.shape}, range [{out.float().min():.2f}, {out.float().max():.2f}]")

mem = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used : {mem:.2f} GB")

## 9 · Train

In [ ]:
best_ckpt = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = EPOCHS,
    lr           = LR,
    ckpt_dir     = CKPT_DIR,
    device       = DEVICE,
)

## 10 · Test Set Evaluation

In [ ]:
print("Loading best checkpoint...")
ckpt = torch.load(best_ckpt, map_location="cpu")
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE)
print(f"Loaded from epoch {ckpt['epoch']} | val MAE {ckpt['val_metrics']['mae']:.4f}")

print("Loading test split...")
test_ds     = ESM2StabilityDataset(DATA_PATH, tokenizer, split="test")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2,
                         shuffle=False, num_workers=0, pin_memory=True)

model.eval()
preds, targets = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(ids, mask)
        preds.extend(out.float().cpu().numpy())
        targets.extend(batch["label"].numpy())

preds, targets = np.array(preds), np.array(targets)
errors  = preds - targets
mae     = float(np.mean(np.abs(errors)))
rmse    = float(np.sqrt(np.mean(errors**2)))
pr      = float(pearsonr(preds, targets)[0])
sr      = float(spearmanr(preds, targets)[0])
r2      = float(1 - np.var(errors) / np.var(targets))
dir_acc = float(np.mean((preds > 0) == (targets > 0)))

print("\n" + "="*52)
print("  TEST SET RESULTS  (3,282 held-out sequences)")
print("="*52)
print(f"  MAE                : {mae:.4f} kcal/mol")
print(f"  RMSE               : {rmse:.4f} kcal/mol")
print(f"  R²                 : {r2:.4f}")
print(f"  Pearson r          : {pr:.4f}")
print(f"  Spearman ρ         : {sr:.4f}")
print(f"  Direction accuracy : {dir_acc:.1%}")
print("="*52)
print(f"  Within ±0.5 kcal/mol : {np.mean(np.abs(errors)<0.5):.1%}")
print(f"  Within ±1.0 kcal/mol : {np.mean(np.abs(errors)<1.0):.1%}")
print(f"  Within ±2.0 kcal/mol : {np.mean(np.abs(errors)<2.0):.1%}")
print(f"\n  vs SaProtΔG (paper) : Pearson ~0.88, RMSE 0.80")
print(f"  vs ESM3ΔG   (paper) : Pearson ~0.87, RMSE 0.80")
print(f"  Gap to SOTA         : {0.88-pr:+.3f} Pearson r")

## 11 · Download Checkpoint to Mac

In [ ]:
from google.colab import files

print(f"Checkpoint saved to Google Drive: {best_ckpt}")
print("Downloading to your Mac...")
files.download(str(best_ckpt))
print()
print("Next step: copy the downloaded file to:")
print("  MLProject/api/models/best_model_esm2.pt")
print("Then restart the API — it will auto-load ESM2.")

## 12 · Quick Inference Test

In [ ]:
def stability_label(dg):
    if dg >  3.0: return "highly stable"
    if dg >  0.5: return "stable"
    if dg > -0.5: return "marginally stable"
    if dg > -3.0: return "unstable"
    return "highly unstable"

def predict(seq):
    model.eval()
    enc = tokenizer(seq, max_length=MAX_LEN, padding="max_length",
                    truncation=True, return_tensors="pt")
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
        dg = model(enc["input_ids"].to(DEVICE),
                   enc["attention_mask"].to(DEVICE)).item()
    return round(dg, 4), stability_label(dg)

samples = [
    ("Stable",   "DEEPAAKNNETKAEKFIRLGEYRMNKAIDAIGRLEHLANRSAYEYTAEQVEAMFGALENKVADVKAKFNT"),
    ("Unstable", "ISGDYAADQEAITKLLQGVSAFIADAGPAVEAGYLPLEAAKAMIMAAVRRFKLGREVEDALDMIG"),
]
print(f"{'Name':<12} {'ΔG (kcal/mol)':>14}  Label")
print("-" * 40)
for name, seq in samples:
    dg, label = predict(seq)
    print(f"{name:<12} {dg:>14.4f}  {label}")